In [28]:
import os
from dotenv import load_dotenv
from langchain.chat_models import init_chat_model

load_dotenv()

os.environ["GROQ_API_KEY"]=os.getenv("GROQ_API_KEY")

model=init_chat_model("groq:qwen/qwen3-32b")

model

ChatGroq(client=<groq.resources.chat.completions.Completions object at 0x16a1fad90>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x16a1fbfd0>, model_name='qwen/qwen3-32b', model_kwargs={}, groq_api_key=SecretStr('**********'), groq_api_base=None, groq_proxy=None)

In [29]:
from typing import TypedDict, Optional

# Define the structure of the QA state
class QAState(TypedDict):
    question: Optional[str]
    context: Optional[str]
    answer: Optional[str]

In [30]:
## input node
def input_validation_node(state):
    question=state.get("question","").strip()

    if not question:
        return {"valid": False, "error": "Question cannot be empty."}
    return {"valid": True}

In [31]:
## context provider node
def context_provider_node(state):
    question=state.get("question","").strip()

    if "LangGraph" in question or "guided project" in question:
        context = (
            "This guided project is about using LangGraph, a Python library to design state-based workflows. "
            "LangGraph simplifies building complex applications by connecting modular nodes with conditional edges."
        )
        return {"context": context}

    return {"context":None}

In [32]:
## llm qa node
def llm_qa_node(state):
    question=state.get("question","").strip()
    context=state.get("context",None)

    if not context:
        return {"answer": "I don't have enough context to answer your question."}

    prompt = f"Context: {context}\nQuestion: {question}\nAnswer the question based on the provided context."

    try:
        response = model.invoke(prompt)
        return {"answer": response.content.strip()}
    except Exception as e:
        return {"answer": f"An error occurred: {str(e)}"}

In [33]:
from langgraph.graph import StateGraph
from langgraph.graph import END

qa_workflow = StateGraph(QAState)

In [34]:
qa_workflow.add_node("InputNode", input_validation_node)
qa_workflow.add_node("ContextNode", context_provider_node)
qa_workflow.add_node("QANode", llm_qa_node)

In [35]:
qa_workflow.set_entry_point("InputNode")

In [36]:
qa_workflow.add_edge("InputNode", "ContextNode")
qa_workflow.add_edge("ContextNode", "QANode")
qa_workflow.add_edge("QANode", END)

In [37]:
qa_app = qa_workflow.compile()

In [38]:
qa_app.invoke({"question": "What is the weather today?"})

{'question': 'What is the weather today?',
 'context': None,
 'answer': "I don't have enough context to answer your question."}

In [39]:
qa_app.invoke({"question": "What is LangGraph?"})

{'question': 'What is LangGraph?',
 'context': 'This guided project is about using LangGraph, a Python library to design state-based workflows. LangGraph simplifies building complex applications by connecting modular nodes with conditional edges.',
 'answer': '<think>\nOkay, the user is asking, "What is LangGraph?" and they provided some context about a guided project using it. Let me start by looking at the context given. The context mentions that LangGraph is a Python library for designing state-based workflows. It helps in building complex applications by connecting modular nodes with conditional edges.\n\nSo, I need to define LangGraph based on this. The answer should include that it\'s a Python library, its purpose is for state-based workflows, and it simplifies complex applications through modular nodes and conditional connections. I should make sure not to add any extra information beyond what\'s provided. Let me check if there\'s anything else in the context. The project is abo